In [ ]:
%%capture
# 1. Environment & Setup
!pip install transformers torch accelerate matplotlib

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Cihaz Tespiti (Device Configuration)
device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan Donanım Hızlandırıcı: {device_str.upper()}\n")

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to(
    device_str
)

# ========================================== #
# 2. Chapter 2/2: Behind the Pipeline & Model Internals
# ========================================== #
print("===")
print("BÖLÜM 2/2: PIPELINE'IN ARKASINDA NE VAR? (TOKENIZER -> MODEL -> POST-PROCESSING)")
print("===")

raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

# 1. Adım: Tokenization (Sayısal Matrise Dönüşüm)
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt").to(
    device_str
)
print("1. Tokenizer Çıktısı (input_ids):\n", inputs["input_ids"])
print("1. Attention Mask:\n", inputs["attention_mask"], "\n")

# 2. Adım: Modele İletim & Ham Logits Çıktısı
with torch.no_grad():
  outputs = model(**inputs)
print("2. Ham Model Çıktısı (Logits):\n", outputs.logits, "\n")

# 3. Adım: Post-processing (Softmax ile Olasılık Hesabı)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print("3. Normalize Edilmiş Olasılıklar (Softmax):\n", predictions)
for i, sentence in enumerate(raw_inputs):
  neg_score, pos_score = predictions[i].tolist()
  label = "POSITIVE" if pos_score > neg_score else "NEGATIVE"
  score = max(neg_score, pos_score)
  print(f"Cümle: '{sentence}' -> Tahmin: {label} ({score:.4f})")
print()

# ========================================== #
# 3. Chapter 2/4: Tokenizer Stratejileri Karşılaştırması
# ========================================== #
print("===")
print("BÖLÜM 2/4: TOKENIZER STRATEJİLERİ KARŞILAŞTIRMASI")
print("===")
comparison_summary = """
| Strateji Türü       | Temel Çalışma Mantığı                        | Örnek Algoritmalar | Avantajı / Dezavantajı                           |
|---------------------|----------------------------------------------|--------------------|--------------------------------------------------|
| Word-based          | Boşluk ve noktalama ile tam kelime bölme     | Python split()     | Basit / Devasa sözlük, yüksek [UNK] riski        |
| Character-based     | Tek tek harflere ve karakterlere bölme       | Karakter Sözlüğü   | Küçük sözlük / Anlamsız tokenlar, çok uzun dizi  |
| Subword-based       | Sık kelimeler bütün, nadir olanlar alt parça | WordPiece, BPE     | Dengeli sözlük, sıfır [UNK], morfolojik koruma   |
"""
print(comparison_summary)

# Alt Kelime (Subword) Ayrıştırma Demosu
sample_word = "tokenization"
tokens = tokenizer.tokenize(sample_word)
ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Örnek Kelime: '{sample_word}'")
print(f"Ayrıştırılan Alt Kelimeler (Tokens): {tokens}")
print(f"Sözlük İndeks Karşılıkları (IDs): {ids}\n")

# ========================================== #
# 4. Chapter 2/5: Handling Multiple Sequences & Attention Mask
# ========================================== #
print("===")
print("BÖLÜM 2/5: ÇOKLU DİZİLER, PADDING VE ATTENTION MASK")
print("===")

seq1 = "I love machine learning."
tokens1 = tokenizer(seq1)["input_ids"]

# Referans Logits (Tekil Çalıştırma)
single_tensor = torch.tensor([tokens1]).to(device_str)
with torch.no_grad():
  ref_logits = model(single_tensor).logits

# Kasıtlı Eksik Maske: Sonuna pad_token_id (0) eklenmiş ama attention mask verilmemiş girdi
pad_id = tokenizer.pad_token_id
bad_padded_tokens = tokens1 + [pad_id, pad_id, pad_id]
bad_tensor = torch.tensor([bad_padded_tokens]).to(device_str)
with torch.no_grad():
  bad_logits = model(bad_tensor).logits

# Doğru Maskelenmiş Girdi (Padding olan yerler 0 yapılır)
real_mask = torch.tensor([[1] * len(tokens1) + [0, 0, 0]]).to(device_str)
with torch.no_grad():
  fixed_logits = model(bad_tensor, attention_mask=real_mask).logits

print(
    f"Referans Logits (Tek Cümle)       : {ref_logits.cpu().numpy().tolist()}"
)
print(
    f"Maskesiz Padding Logits (SAPMA!)  : {bad_logits.cpu().numpy().tolist()}"
)
print(
    f"Doğru Maskelenmiş Padding Logits   : {fixed_logits.cpu().numpy().tolist()}"
)
print(
    "-> ÇIKARIM: Attention Mask kullanılmadığında dolgu (pad) tokenları tahmin değerlerini bozar!\n"
)

# ========================================== #
# 5. Görselleştirme: Attention Mask'in Logits Üzerindeki Etkisi
# ========================================== #
labels = ["Negative", "Positive"]
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
rects1 = ax.bar(
    x - width,
    ref_logits.cpu().numpy()[0],
    width,
    label="Referans (Tekil)",
    color="#2ca02c",
)
rects2 = ax.bar(
    x,
    bad_logits.cpu().numpy()[0],
    width,
    label="Maskesiz Padding (Hatalı)",
    color="#d62728",
)
rects3 = ax.bar(
    x + width,
    fixed_logits.cpu().numpy()[0],
    width,
    label="Maskeli Padding (Düzeltilmiş)",
    color="#1f77b4",
)

ax.set_ylabel("Ham Çıktı (Logits)")
ax.set_title(
    "Attention Mask'in Model Çıktısına Etkisi (Bölüm 2 Adli Test Analizi)"
)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()